In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from shapely.geometry import box

# Zones to plot (Brest + Australia)
zones = [
    {"name": "Brest", "min_lon": -6.25, "max_lon": -6, "min_lat": 46.6, "max_lat": 47},
    {"name": "Adelaide Offshore", "min_lon": 137, "max_lon": 141, "min_lat": 33, "max_lat": 36}
]

# Create plot with PlateCarree projection
fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()

# Add land, ocean, and borders for context
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.COASTLINE)

# Plot rectangles + red points for each zone
for zone in zones:
    rect = box(zone["min_lon"], zone["min_lat"], zone["max_lon"], zone["max_lat"])
    ax.add_geometries([rect], crs=ccrs.PlateCarree(),
                      edgecolor='red', facecolor='none', linewidth=2)
    
    # Compute center and plot red point
    center_lon = (zone["min_lon"] + zone["max_lon"]) / 2
    center_lat = (zone["min_lat"] + zone["max_lat"]) / 2
    ax.plot(center_lon, center_lat, 'ro', markersize=6, transform=ccrs.PlateCarree())

# Add title
plt.title("Earth Map with Highlighted Zones (Brest & Australia)", fontsize=14)

plt.show()

### Copernicus data 

In [ ]:
import os
import re

base_path = './DataOcean/'

# Regular expression to match a date in the format YYYY-MM-DD
date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')

dates_found = []

# Walk through all folders after DataOcean
for root, dirs, files in os.walk(base_path):
    for dir_name in dirs:
        match = date_pattern.search(dir_name)
        if match:
            dates_found.append(match.group())

# Remove duplicates and sort
dates_found = sorted(set(dates_found))

print(dates_found)


In [ ]:
import os
import re

import pandas as pd

# Base path
base_path = './DataOcean/'
zone_name = 'brest'

# File names
copernicus_files = {
    'wind': ['wind_latlong.csv', 'wind_cart.csv'],
    'uwCurrent': ['uwCurrent_latlong.csv', 'uwCurrent_cart.csv'],
    'bathy': ['bathy_latlong.csv', 'bathy_cart.csv']
}

# Regular expression to match dates in folder names YYYY-MM-DD
date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')

# Store results for each date
all_copernicus_data = {}

# Loop through folders under DataOcean
for root, dirs, files in os.walk(base_path):
    for dir_name in dirs:
        match = date_pattern.search(dir_name)
        if match:
            date_str = match.group()
            save_path_copernicus = os.path.join(base_path, dir_name, zone_name, 'copernicus/')
            
            if os.path.exists(save_path_copernicus):
                date_data = {}
                
                # Load each file if it exists
                for key, file_list in copernicus_files.items():
                    for file_name in file_list:
                        file_path = os.path.join(save_path_copernicus, file_name)
                        if os.path.exists(file_path):
                            df = pd.read_csv(file_path)
                            
                            # Special processing for 'cart' files with 'time' column
                            if 'cart' in file_name and 'time' in df.columns:
                                df['date'] = pd.to_datetime(df['time']).dt.date
                                df['time'] = pd.to_datetime(df['time']).dt.time
                                
                                # Example: Filter out start date (you can modify as needed)
                                # start_dateTime_dt = pd.to_datetime(start_dateTime).normalize().date()
                                # df = df[df['date'] != start_dateTime_dt]
                                
                            date_data[file_name] = df
                
                all_copernicus_data[date_str] = date_data

In [ ]:
# Example: Accessing wind_cart.csv
wind_cart_df = all_copernicus_data[example_date]['wind_cart.csv']
print(wind_cart_df.head())

In [ ]:
# Example: Accessing uwCurrent_cart.csv for a specific date
example_date = list(all_copernicus_data.keys())[0]
uw_current_cart_df = all_copernicus_data[example_date]['uwCurrent_cart.csv']
print(uw_current_cart_df.head())

In [ ]:
# Dictionary to store stats per date
date_stats = {}

for date_str, files in all_copernicus_data.items():
    stats = {}
    
    # Process uwCurrent_cart.csv (ocean currents)
    if 'uwCurrent_cart.csv' in files:
        df_current = files['uwCurrent_cart.csv']
        stats.update({
            'uo_min': df_current['uo'].min(),
            'uo_max': df_current['uo'].max(),
            'uo_mean': df_current['uo'].mean(),
            'vo_min': df_current['vo'].min(),
            'vo_max': df_current['vo'].max(),
            'vo_mean': df_current['vo'].mean()
        })
    
    # Process wind_cart.csv (wind components)
    if 'wind_cart.csv' in files:
        df_wind = files['wind_cart.csv']
        stats.update({
            'northward_wind_min': df_wind['northward_wind'].min(),
            'northward_wind_max': df_wind['northward_wind'].max(),
            'northward_wind_mean': df_wind['northward_wind'].mean(),
            'eastward_wind_min': df_wind['eastward_wind'].min(),
            'eastward_wind_max': df_wind['eastward_wind'].max(),
            'eastward_wind_mean': df_wind['eastward_wind'].mean()
        })
    
    date_stats[date_str] = stats

# Convert to DataFrame for easy viewing
date_stats_df = pd.DataFrame(date_stats).T.sort_index()
print("Statistics per date:")
print(date_stats_df)

# Compute global statistics across all dates
all_current_dfs = [files['uwCurrent_cart.csv'] for files in all_copernicus_data.values() if 'uwCurrent_cart.csv' in files]
all_wind_dfs = [files['wind_cart.csv'] for files in all_copernicus_data.values() if 'wind_cart.csv' in files]

global_stats = {}

if all_current_dfs:
    global_current_df = pd.concat(all_current_dfs, ignore_index=True)
    global_stats.update({
        'uo_min': global_current_df['uo'].min(),
        'uo_max': global_current_df['uo'].max(),
        'uo_mean': global_current_df['uo'].mean(),
        'vo_min': global_current_df['vo'].min(),
        'vo_max': global_current_df['vo'].max(),
        'vo_mean': global_current_df['vo'].mean()
    })

if all_wind_dfs:
    global_wind_df = pd.concat(all_wind_dfs, ignore_index=True)
    global_stats.update({
        'northward_wind_min': global_wind_df['northward_wind'].min(),
        'northward_wind_max': global_wind_df['northward_wind'].max(),
        'northward_wind_mean': global_wind_df['northward_wind'].mean(),
        'eastward_wind_min': global_wind_df['eastward_wind'].min(),
        'eastward_wind_max': global_wind_df['eastward_wind'].max(),
        'eastward_wind_mean': global_wind_df['eastward_wind'].mean()
    })

print("\nGlobal statistics across all dates:")
print(global_stats)
